In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import crosscoders as xc
import torch


------------------------- CONSTANTS -------------------------
GlobalsConfig(
    PROJECT_ROOT_DIR = '/home/yandy/repos/crosscoders',
    CONFIG_FILEPATH = '/home/yandy/repos/crosscoders/src/scripts/configs/train.yml',
    DATA_DIR = '/home/ec2-user/crosscoders/data',
    EXPERIMENT = ExperimentConfig(
        BATCH_SIZE = 8192,
        MAX_RECORDS = None,
        MAX_BATCHES = 1000000,
        MAX_TOKENS = 1000000,
        NUM_GPUS = 1,
        NUM_TRAINERS = 1,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
)
-------------------------------------------------------------



In [3]:
# xc.dataclasses.configs.RunnerConfig(xc.dataclasses.configs.ModelConfig('acausal'))

In [6]:
CONSTANTS.CONFIG_FILEPATH = '/home/yandy/repos/crosscoders/src/scripts/configs/train.yml'

In [4]:

from crosscoders.constants import CONSTANTS
from crosscoders.dataclasses.configs.runner import RunnerConfig
from crosscoders.utils import from_dict, get_config


runner_cfg = from_dict(
    RunnerConfig,
    get_config(CONSTANTS.CONFIG_FILEPATH).get('RUNNER', {})
)
runner_cfg

RunnerConfig(
    MODEL = ModelConfig(
        CAUSALITY = 'acausal',
        LOCALITY = 'global',
        ACTIVATION_FUNCTION = 'jumprelu',
        eps = 2,
        N_LAYERS = 12,
        D_MODEL = 768,
        D_CODER = 16384,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
    LOSS = LossConfig(
        c = 4,
        lambda_s = 10,
        lambda_p = 3e-06,
    ),
    OPTIMIZER = OptimizerConfig(
        optimizer = <class 'torch.optim.adam.Adam'>,
        parameters = OptimizerParameters(
            lr = 0.0004,
            betas = (0.9, 0.999),
            fused = True,
        ),
    ),
)

In [5]:
bucket = 'crosscoders'



import os
import boto3
import pathlib


def get_model_checkpoint(path, local=True):


    if not local:
        checkpoint_path = f'/tmp/crosscoders/checkpoints/{path}'
        checkpoint_path = pathlib.Path(checkpoint_path)
        os.makedirs(checkpoint_path.parent, exist_ok=True)

        with open(checkpoint_path, 'wb') as infl:
            boto3.resource('s3') \
                .Object(bucket, path) \
                .download_fileobj(infl)


    with open(checkpoint_path, 'rb') as infl:
        checkpoint_dict = torch.load(infl)


    return checkpoint_dict

In [6]:
checkpoint_dict = get_model_checkpoint(
    'ray/tune/jumprelu/uniform_init/trained/TorchTrainer_2025-02-19_02-25-39/TorchTrainer_d041a_00000_0_2025-02-19_02-25-40/checkpoint_000000/model.pt',
    False)

In [7]:
from crosscoders.autoencoders.acausal.model import AcausalAutoencoder


def load_model_from_checkpoint(checkpoint_dict):

    model = AcausalAutoencoder(runner_cfg.MODEL)

    model.load_state_dict(checkpoint_dict)
    model.to(runner_cfg.MODEL.HARDWARE.device)
    model.eval()

    return model


model = load_model_from_checkpoint(checkpoint_dict)

In [8]:
model.W_dec.shape

torch.Size([16384, 12, 768])

In [9]:
import datasets
import ray
# from crosscoders.data.dataset import RayDataset
from crosscoders.data.preprocessing import TokenToActivations


# train_ds = TinyStoriesRayDataset().load('activations')
hf_dataset = datasets.load_dataset('roneneldan/TinyStories', streaming=True)
train_ds = ray.data.from_huggingface(hf_dataset['train'], concurrency=1)


train_ds_ = train_ds.map_batches(
    TokenToActivations,
    batch_size=8,
    concurrency=1,
    num_gpus=1,
    # num_cpus=1
)

train_dl = train_ds_.iter_torch_batches(
    # batch_size=CONSTANTS.EXPERIMENT.BATCH_SIZE,
    batch_size=500,
    # local_shuffle_buffer_size=16
    device='cuda'
)


2025-02-20 17:33:51,450	INFO worker.py:1654 -- Connecting to existing Ray cluster at address: 172.27.88.212:6379...
2025-02-20 17:33:51,463	INFO worker.py:1832 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 


(_MapWorker pid=12738) 
(_MapWorker pid=12738) ------------------------- CONSTANTS -------------------------
(_MapWorker pid=12738) GlobalsConfig(
(_MapWorker pid=12738)     PROJECT_ROOT_DIR = '/home/yandy/repos/crosscoders',
(_MapWorker pid=12738)     CONFIG_FILEPATH = '/home/yandy/repos/crosscoders/src/scripts/configs/train.yml',
(_MapWorker pid=12738)     DATA_DIR = '/home/ec2-user/crosscoders/data',
(_MapWorker pid=12738)     EXPERIMENT = ExperimentConfig(
(_MapWorker pid=12738)         BATCH_SIZE = 8192,
(_MapWorker pid=12738)         MAX_RECORDS = None,
(_MapWorker pid=12738)         MAX_BATCHES = 1000000,
(_MapWorker pid=12738)         MAX_TOKENS = 1000000,
(_MapWorker pid=12738)         NUM_GPUS = 1,
(_MapWorker pid=12738)         NUM_TRAINERS = 1,
(_MapWorker pid=12738)         HARDWARE = HardwareConfig(
(_MapWorker pid=12738)             dtype = torch.float32,
(_MapWorker pid=12738)             device = 'cuda',
(_MapWorker pid=12738)         ),
(_MapWorker pid=12738)     ),
(

In [10]:
for batch_idx, batch in enumerate(train_dl):

    # loss = runner.training_step(batch)

    # print(loss)

    if batch_idx == 10:
        break

2025-02-20 17:33:51,590	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-20_17-19-37_143929_12218/logs/ray-data
2025-02-20 17:33:51,591	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadHuggingFace] -> ActorPoolMapOperator[MapBatches(TokenToActivations)]


Running 0: 0.00 row [00:00, ? row/s]

- ReadHuggingFace->SplitBlocks(15) 1: 0.00 row [00:00, ? row/s]

- MapBatches(TokenToActivations) 2: 0.00 row [00:00, ? row/s]

In [11]:
batch.keys()
batch_idx

10

In [12]:
x = batch['gpt2-small.resid_post']
x_hat = model(x)

In [25]:
model.x_enc.max(-1)

torch.return_types.max(
values=tensor([221.4532, 242.1108, 261.0635, 268.5757, 237.0896, 267.9719, 278.2173,
        223.4019, 277.1419, 287.2440, 213.0963, 226.5411, 242.0563, 212.7635,
        295.4177, 314.7568, 267.8117, 248.5595, 271.3666, 218.4173, 222.9550,
        240.7592, 233.5256, 238.4170, 273.2147, 266.8994, 261.5863, 206.3231,
        298.3366, 307.1201, 259.6764, 276.3639], device='cuda:0',
       grad_fn=<MaxBackward0>),
indices=tensor([13589, 10288, 10288, 10288, 10288,  9685,  4302,  2744,  4302, 10288,
         4302,  4302,  4302,  4302, 12755,  9685,  2744,  9685,  4302,  2744,
         9685,  4302,  4302,  2744,  2744,  9685,  2744,  4302,  9685,  9685,
         9685,  9685], device='cuda:0'))

In [13]:
from transformer_lens import HookedTransformer

gpt2 = HookedTransformer.from_pretrained('gpt2-small')


Loaded pretrained model gpt2-small into HookedTransformer


In [14]:
tokens = [gpt2.to_string(_) for _ in batch['tokens'].to(int)]
tokens

[' for',
 ' later',
 '.',
 '\n',
 '\n',
 'Tom',
 ' went',
 ' to',
 ' play',
 ' with',
 ' his',
 ' friend',
 ',',
 ' Sam',
 '.',
 ' They',
 ' played',
 ' a',
 ' game',
 ' where',
 ' they',
 ' had',
 ' to',
 ' escape',
 ' from',
 ' a',
 ' pretend',
 ' monster',
 '.',
 ' They',
 ' ran',
 ' and',
 ' hid',
 ',',
 ' but',
 ' the',
 ' monster',
 ' always',
 ' found',
 ' them',
 '.',
 ' Tom',
 ' felt',
 ' scared',
 ',',
 ' but',
 ' he',
 ' remembered',
 ' his',
 ' special',
 ' gum',
 '.',
 ' He',
 ' thought',
 ' it',
 ' could',
 ' help',
 ' them',
 ' win',
 ' the',
 ' game',
 '.',
 '\n',
 '\n',
 'Tom',
 ' took',
 ' out',
 ' the',
 ' thin',
 ' gum',
 ' and',
 ' shared',
 ' it',
 ' with',
 ' Sam',
 '.',
 ' They',
 ' both',
 ' che',
 'wed',
 ' the',
 ' gum',
 ' and',
 ' started',
 ' to',
 ' run',
 ' from',
 ' the',
 ' monster',
 ' again',
 '.',
 ' But',
 ' this',
 ' time',
 ',',
 ' they',
 ' did',
 ' not',
 ' escape',
 '.',
 ' The',
 ' gum',
 ' made',
 ' them',
 ' feel',
 ' sick',
 ' and',
 ' slo

In [ ]:
ttl = list(zip(tokens, model.x_enc.max(-1).indices.cpu().numpy())

[(' for', 4302),
 (' later', 4302),
 ('.', 9685),
 ('\n', 9685),
 ('\n', 10288),
 ('Tom', 9685),
 (' went', 9685),
 (' to', 9685),
 (' play', 4302),
 (' with', 4302),
 (' his', 2744),
 (' friend', 4302),
 (',', 4302),
 (' Sam', 2744),
 ('.', 9685),
 (' They', 9685),
 (' played', 4302),
 (' a', 4302),
 (' game', 4302),
 (' where', 4302),
 (' they', 9685),
 (' had', 4302),
 (' to', 9685),
 (' escape', 4302),
 (' from', 4302),
 (' a', 10288),
 (' pretend', 12755),
 (' monster', 4302),
 ('.', 12755),
 (' They', 9685),
 (' ran', 9685),
 (' and', 9685),
 (' hid', 9685),
 (',', 9685),
 (' but', 9685),
 (' the', 2744),
 (' monster', 9685),
 (' always', 9685),
 (' found', 4302),
 (' them', 9685),
 ('.', 9685),
 (' Tom', 9685),
 (' felt', 9685),
 (' scared', 9685),
 (',', 9685),
 (' but', 9685),
 (' he', 9685),
 (' remembered', 9685),
 (' his', 2744),
 (' special', 4302),
 (' gum', 2744),
 ('.', 9685),
 (' He', 9685),
 (' thought', 9685),
 (' it', 9685),
 (' could', 9685),
 (' help', 2744),
 (' 

In [16]:
sorted(ttl, key=lambda _: _[1])

[(' his', 2744),
 (' Sam', 2744),
 (' the', 2744),
 (' his', 2744),
 (' gum', 2744),
 (' help', 2744),
 (' win', 2744),
 (' game', 2744),
 (' thin', 2744),
 (' with', 2744),
 (' the', 2744),
 (' the', 2744),
 (' this', 2744),
 (',', 2744),
 (' The', 2744),
 (' made', 2744),
 (' The', 2744),
 (' the', 2744),
 (' never', 2744),
 (' up', 2744),
 (' had', 2744),
 (' wore', 2744),
 (' the', 2744),
 (' to', 2744),
 (' rain', 2744),
 (' made', 2744),
 (' his', 2744),
 (' vest', 2744),
 (' wet', 2744),
 (' wet', 2744),
 (' his', 2744),
 (' saw', 2744),
 (' his', 2744),
 (' wet', 2744),
 (' wet', 2744),
 (' favorite', 2744),
 (' was', 2744),
 (' wet', 2744),
 (' dry', 2744),
 (' the', 2744),
 (' sun', 2744),
 (' was', 2744),
 (' very', 2744),
 (' hot', 2744),
 (' and', 2744),
 (' water', 2744),
 (' because', 2744),
 (' boat', 2744),
 (',', 2744),
 (' you', 2744),
 (' sad', 2744),
 (' The', 2744),
 (' water', 2744),
 (' and', 2744),
 (' cannot', 2744),
 (' float', 2744),
 ('.', 2744),
 (' miss',

In [13]:
cfg

RunnerConfig(
    MODEL = ModelConfig(
        CAUSALITY = 'acausal',
        LOCALITY = 'global',
        N_LAYERS = 12,
        D_MODEL = 768,
        D_CODER = 16384,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
    LOSS = AcausalLossConfig(
        L1_COEFFICIENT = 1.0,
    ),
    OPTIMIZER = OptimizerConfig(
        optimizer = <class 'torch.optim.adam.Adam'>,
        parameters = OptimizerParameters(
            lr = 0.0002,
            betas = (0.9, 0.999),
            fused = True,
        ),
    ),
)